# Frozen Laya REAL / NOISY / NORMALIZED benchmark

This notebook performs the one-time official test evaluation after the Laya
checkpoint is ready. The **same frozen classifier** is used for all conditions:

`REAL = official test`, `NOISY = paired M1 synthetic corruption`,
`NORMALIZED = exactly NOISY passed through the selected TNT Hybrid V3 model`.

The execution switch is disabled by default to keep official test frozen until
all paths and manifests have been reviewed.

In [1]:
from pathlib import Path
import json, sys

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/downstream").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the repository checkout")

REPO_ROOT = find_repository_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data/external/injongo_intent_wol"
TEST_PATH = DATA_DIR / "test.jsonl"
LAYA_CHECKPOINT = REPO_ROOT / "checkpoints/downstream/laya_normalization_v1"
OUTPUT_DIR = REPO_ROOT / "results/downstream/laya_normalization_v1"
SEED = 2026

DEVICE = None            # None selects CUDA when available
print("Laya checkpoint:", LAYA_CHECKPOINT)
print("Results:", OUTPUT_DIR)

Laya checkpoint: c:\Users\thioy\Desktop\School\PFE\checkpoints\downstream\laya_normalization_v1
Results: c:\Users\thioy\Desktop\School\PFE\results\downstream\laya_normalization_v1


## Readiness and locked TNT selection

The normalizer is not retrained. The repository's locked V3 selection is the
M1-pretrained checkpoint adapted on Gold train with learning rate 5e-5 and
`word_keep_threshold=0.50`, selected on Gold-dev WER. Hashes are checked before
inference. Its local checkpoint must be present because large weights are not
stored in Git.

In [2]:
from src.downstream.tnt_normalizer import resolve_selected_tnt_checkpoint

assert LAYA_CHECKPOINT.is_dir(), "Run the fine-tuning notebook first."
training_manifest = json.loads((LAYA_CHECKPOINT / "training_manifest.json").read_text())
assert training_manifest["test_used"] is False
tnt_checkpoint, tnt_selection = resolve_selected_tnt_checkpoint(REPO_ROOT)
display({
    "laya_ready": True,
    "laya_best_epoch": training_manifest["best_epoch"],
    "tnt_checkpoint": str(tnt_checkpoint),
    "tnt_candidate": tnt_selection["candidate"],
    "tnt_selection_split": tnt_selection["selection_split"],
    "tnt_dev_metrics": tnt_selection["development_metrics"],
})

{'laya_ready': True,
 'laya_best_epoch': 4,
 'tnt_checkpoint': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\checkpoints\\m7_seq2seq\\tnt_hybrid_v3_optimization_v1\\m1_v3\\seed_2026\\gold_adapt_lr_5e5_word_keep_t050\\attempt_001\\best_model',
 'tnt_candidate': 'm1_v3_gold_adapt_lr_5e5_word_keep_t050',
 'tnt_selection_split': 'Gold-dev',
 'tnt_dev_metrics': {'rows': 30,
  'cer': 0.1706408345752608,
  'wer': 0.5728813559322034,
  'chrf': 64.62921378272632,
  'exact_match': 0.0,
  'correction_tp': 3,
  'correction_fp': 57,
  'correction_fn': 69,
  'correction_precision': 0.05,
  'correction_recall': 0.041666666666666664,
  'correction_f1': 0.04545454545454545,
  'overcorrection_rate': 0.95,
  'unchanged_reference_rows': 0,
  'unchanged_sentence_overcorrection_rate': 0.0}}

## One-time official test benchmark

Only this cell loads `test.jsonl`. It creates paired texts, normalizes NOISY in
batches, evaluates the same calibrated Laya checkpoint on all three conditions,
then writes metrics, per-class reports, confusion matrices, predictions and the
four-way paired analysis. No model is updated.

In [3]:
from src.downstream.injongo import load_split
from src.downstream.benchmark import run_frozen_benchmark

RUN_FROZEN_TEST = True  # review readiness below, then change once to True

if RUN_FROZEN_TEST:
    test_rows = load_split(TEST_PATH, "test")
    benchmark = run_frozen_benchmark(
        test_rows,
        laya_checkpoint=LAYA_CHECKPOINT,
        tnt_checkpoint=tnt_checkpoint,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        device=DEVICE,
    )
    display(benchmark["metrics"])
else:
    print("Frozen test remains unopened. Set RUN_FROZEN_TEST=True when the pipeline is final.")

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


{'REAL': {'rows': 638,
  'accuracy': 0.8087774294670846,
  'macro_f1': 0.8117716357020083},
 'NOISY': {'rows': 638,
  'accuracy': 0.48589341692789967,
  'macro_f1': 0.4596851589967839},
 'NORMALIZED': {'rows': 638,
  'accuracy': 0.603448275862069,
  'macro_f1': 0.6087104006914915},
 'deltas': {'accuracy': {'real': 0.8087774294670846,
   'noisy': 0.48589341692789967,
   'normalized': 0.603448275862069,
   'noise_drop': 0.322884012539185,
   'normalization_gain': 0.11755485893416928,
   'recovery_rate': 0.3640776699029126},
  'macro_f1': {'real': 0.8117716357020083,
   'noisy': 0.4596851589967839,
   'normalized': 0.6087104006914915,
   'noise_drop': 0.35208647670522436,
   'normalization_gain': 0.1490252416947076,
   'recovery_rate': 0.423263179799647}},
 'paired': {'rows': 638,
  'help_rate': 0.1724137931034483,
  'harm_rate': 0.054858934169279,
  'unchanged_correct_rate': 0.43103448275862066,
  'unresolved_rate': 0.34169278996865204,
  'counts': {'HELPED': 110,
   'UNCHANGED_CORRECT':

## Inspect saved results

This section is read-only and becomes available after the benchmark cell.
`recovery_rate` is emitted only when REAL outperforms NOISY, so a zero/negative
noise drop is never hidden by an unstable denominator.

In [4]:
import pandas as pd

metrics_path = OUTPUT_DIR / "metrics.json"
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame({condition: values for condition, values in metrics.items() if condition in {"REAL", "NOISY", "NORMALIZED"}}).T)
    display(metrics["deltas"])
    display(metrics["paired"])
    display(pd.read_csv(OUTPUT_DIR / "paired_analysis.csv").head(20))
else:
    print("No benchmark results yet.")

,rows,accuracy,macro_f1
REAL,638.0,0.808777,0.811772
NOISY,638.0,0.485893,0.459685
NORMALIZED,638.0,0.603448,0.608710


{'accuracy': {'real': 0.8087774294670846,
  'noisy': 0.48589341692789967,
  'normalized': 0.603448275862069,
  'noise_drop': 0.322884012539185,
  'normalization_gain': 0.11755485893416928,
  'recovery_rate': 0.3640776699029126},
 'macro_f1': {'real': 0.8117716357020083,
  'noisy': 0.4596851589967839,
  'normalized': 0.6087104006914915,
  'noise_drop': 0.35208647670522436,
  'normalization_gain': 0.1490252416947076,
  'recovery_rate': 0.423263179799647}}

{'rows': 638,
 'help_rate': 0.1724137931034483,
 'harm_rate': 0.054858934169279,
 'unchanged_correct_rate': 0.43103448275862066,
 'unresolved_rate': 0.34169278996865204,
 'counts': {'HELPED': 110,
  'UNCHANGED_CORRECT': 275,
  'UNRESOLVED': 218,
  'HARMED': 35}}

,example_id,row_id,real_text,noisy_text,normalized_text,gold_intent,real_prediction,noisy_prediction,normalized_prediction,category
0,test-00000064,test:alarm:test-00000064,Déffal ab fàttali ci salon bi ci Lagos ci 5: 4...,déffal ab fattali thi salone bi thi lagos thi ...,déffal ab fattali ci saloon bi ci lagos ci 5: ...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
1,test-00000065,test:alarm:test-00000065,Déffal ab fàttali ci téere bi nekk Dubaï ci 7:...,déffal ab fattali thi teré bi nekk doubaï thi ...,déffal ab fattali ci teere bi nekk dubaï ci 7:...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
2,test-00000066,test:alarm:test-00000066,Ndax mën nga tënk ab woote ngir sama ndaje ci ...,ndakh meune nga teunk ab woté nguir sama ndadi...,Ndax mën nga tënk ab woote ngir sama ndadee ci...,alarm,calendar_update,weather,alarm,HELPED
3,test-00000067,test:alarm:test-00000067,Yal nañ sàkk ab waxtaan ci njëlbéen gi ci Sene...,yal nagne sakk ab wakhtane thi njeulbene gui t...,Yaal na ññ sakk ab waxtan ci njëlbee i ci sene...,alarm,weather,alarm,alarm,UNCHANGED_CORRECT
4,test-00000068,test:alarm:test-00000068,Yal nañ sàkk ab woote ci àdduna bi ci Marakech...,yal nagne sakk ab woté thi addouna bi thi mara...,Yaal na ññ sakk ab woté ci adduna bi ci marake...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
5,test-00000069,test:alarm:test-00000069,Ndax mën nga tëral alarm ci waxtu fadjar ci Ndar?,ndakh meune nga teural alarm thi wakhtou faddi...,Ndax mën nga tëral alarm ci waxu faddiar ci ndar?,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
6,test-00000070,test:alarm:test-00000070,Défalma ab fatali ci ab waxtu ci restaurant bi...,défalma ab fatali thi ab wakhtou thi restaoura...,Défalma ab fatali ci ab waxcu ci reestàurant b...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
7,test-00000071,test:alarm:test-00000071,Sàkkal ab waxtaan ci musiwaayu Rabat ci 5:25 c...,sakkal ab wakhtane thi mousiwayou rabat thi 5:...,sakkal ab waxtan ci musiwayu rabat ci 5:25 ci ...,alarm,alarm,alarm,time,HARMED
8,test-00000072,test:alarm:test-00000072,Ndax mën nga tëral alarm ci kër Mamadou ci Dak...,ndakh meune nga teural alarm thi keur mamadoou...,Ndax mën nga tëral alarm ci kër mamadóu ci dak...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
9,test-00000073,test:alarm:test-00000073,Jëfandikoo ab waxtaan ci màrse bi ci Akra ci 8...,jeufandiko ab wakhtane thi marsé bi thi akra t...,jeufandiko ab waxan ci marse bi ci akra ci 8:0...,alarm,alarm,alarm,alarm,UNCHANGED_CORRECT
